# 02 - Raw BLD versus Fixed and Adaptive CCI

This notebook runs a disjoint evaluation cohort with identical model, prompt, seed, masks, and scheduler settings. `disabled` is raw BLD without CCI updates, `fixed_equal` uses CCI with fixed constraint coefficients, and `feedback` enables adaptive dual weighting.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_ROOT = Path('/kaggle/working/cci-diff')
ASSET_ROOT = Path('/kaggle/input/cci-assets')
DATA_ROOT = Path('/kaggle/input/celebamask-hq/CelebAMask-HQ')
MODEL_PATH = ASSET_ROOT / 'sd2-1-base'
CLASSIFIER_PATH = ASSET_ROOT / 'resnet50_multilabel_model.pth'
IDENTITY_MODEL_PATH = ASSET_ROOT / 'facenet_vggface2.ts'
IMAGE_ROOT = DATA_ROOT / 'CelebA-HQ-img'
MASK_ROOT = DATA_ROOT / 'CelebAMask-HQ-mask-anno'
DISCOVERY_IDS_PATH = Path('/kaggle/working/cci_graph_discovery/discovery_ids.json')
OUTPUT_ROOT = Path('/kaggle/working/cci_fixed_vs_adaptive')

DEVICE = 'cuda'
SAMPLE_COUNT = 300
NUM_INFERENCE_STEPS = 35
SEED = 42
FIXED_REGIONS = {
    'smile': ['mouth', 'upper_lip', 'lower_lip'],
    'hair': ['hair'],
}

In [ ]:
required = [PROJECT_ROOT, MODEL_PATH, CLASSIFIER_PATH, IDENTITY_MODEL_PATH, IMAGE_ROOT, MASK_ROOT, DISCOVERY_IDS_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Update the configuration paths; missing: ' + ', '.join(missing))
discovery_ids = json.loads(DISCOVERY_IDS_PATH.read_text())
assert set(discovery_ids) == {'smile', 'hair'}
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT), '--no-deps'], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Evaluation excludes', {task: len(ids) for task, ids in discovery_ids.items()})

In [ ]:
# The pilot's task definitions bind smile to mouth + both lips and hair to hair only.
command = [
    sys.executable, 'scripts/run_clean_cci_pilot.py',
    '--features', 'smile', 'hair', '--limit', str(SAMPLE_COUNT),
    '--controller_modes', 'disabled', 'fixed_equal', 'feedback',
    '--exclude_ids_json', str(DISCOVERY_IDS_PATH),
    '--model_path', str(MODEL_PATH), '--classifier_path', str(CLASSIFIER_PATH),
    '--identity_model_path', str(IDENTITY_MODEL_PATH),
    '--image_root', str(IMAGE_ROOT), '--mask_root', str(MASK_ROOT),
    '--device', DEVICE, '--torch_dtype', 'auto',
    '--python_executable', sys.executable,
    '--seed', str(SEED), '--num_inference_steps', str(NUM_INFERENCE_STEPS),
    '--mask_shapes', '4,4,3', '--continue_on_error',
    '--output_dir', str(OUTPUT_ROOT),
]
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
import pandas as pd
results = pd.read_csv(OUTPUT_ROOT / 'pilot_results.csv')
results['controller_mode'] = results['variant'].map({'A0': 'raw_bld', 'A2': 'fixed_equal', 'A3': 'feedback'})
display(results.groupby(['feature', 'controller_mode']).agg(
    count=('target_pass', 'size'),
    generation_classifier_fr=('target_pass', 'mean'),
    desired_probability=('desired_probability', 'mean'),
    identity_cosine=('identity_cosine', 'mean'),
    non_target_drift=('non_target_drift', 'mean'),
    changed_fraction_5=('changed_fraction_5', 'mean'),
    outside_semantic_fraction_5=('outside_semantic_fraction_5', 'mean'),
    residual_tv=('residual_tv', 'mean'),
    runtime_seconds=('runtime_seconds', 'mean'),
).reset_index())
print('Source/output comparisons:', OUTPUT_ROOT / 'comparisons')